# 🔭 Lab 10: 구독별 거버넌스 & 관측 — App Insights KQL 종합 테스트

## 목적
AI Gateway 가 **여러 구독(테넌트)** 의 토큰·비용·거버넌스 차단을 **하나의 관측 평면**에서
분해해 보여주는지 실제 트래픽 + App Insights KQL 로 검증합니다. (Lab 6 단일 관측 → 멀티 구독 확장)

| 차원 | 소스 | KQL |
|---|---|---|
| Provider × Subscription × Model 토큰 | `customMetrics` (`llm-emit-token-metric`) | Query 1 |
| 구독별 비용 추정 | `customMetrics` | Query 2 |
| 429/403 거버넌스 차단 추이 | `requests` (자동 수집) | Query 3 |
| 구독별 quota 사용률 | `customMetrics` | (저장 쿼리) |

## 사전 조건
- APIM + App Insights 연동, `az login`
- **커스텀 메트릭 활성화** — 이 노트북이 `metrics: true` 를 자동 확인/설정합니다
  (상세: [Lab 6 · 2단계](../lab06-monitoring/README.md))

## 구조
| Phase | 내용 |
|---|---|
| Phase 1 | 2개 구독(team-a/team-b) 라벨 트래픽 생성 (+team-b 는 소한도로 403 유발) |
| Phase 2 | 수집 대기 (customMetrics 5~10분) |
| Phase 3 | KQL 실시간 조회 (구독별 토큰/비용/차단 추이) |
| 정리 | Product/구독 삭제 |


In [1]:
# ─── 환경 설정 + App Insights KQL 하네스 ───
import os, time, json, subprocess, tempfile
from datetime import datetime, timezone
import requests
from dotenv import load_dotenv
load_dotenv("../../.env", override=True)

def az(args):
    r = subprocess.run(["az"] + args, capture_output=True, text=True)
    return r.stdout.strip(), r.stderr.strip(), r.returncode
def az_json(args):
    out, err, rc = az(args)
    if rc != 0 or not out: return None
    try: return json.loads(out)
    except json.JSONDecodeError: return out

SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID") or (az_json(["account","show","--query","id","-o","json"]) or "")
RESOURCE_GROUP  = os.getenv("RESOURCE_GROUP","")
APIM_NAME       = os.getenv("APIM_NAME","")
if not (RESOURCE_GROUP and APIM_NAME):
    apims = az_json(["apim","list","--query","[].{name:name,rg:resourceGroup}","-o","json"]) or []
    if apims:
        APIM_NAME = APIM_NAME or apims[0]["name"]; RESOURCE_GROUP = RESOURCE_GROUP or apims[0]["rg"]
APIM_URL        = os.getenv("APIM_URL") or f"https://{APIM_NAME}.azure-api.net"
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME","gpt-4.1-nano")
API_VERSION     = "2025-04-01-preview"; ARM_API = "2024-06-01-preview"
APP_INSIGHTS_APP_ID = os.getenv("APP_INSIGHTS_APP_ID","")
if not APP_INSIGHTS_APP_ID:
    comp = az_json(["monitor","app-insights","component","show","-g",RESOURCE_GROUP,
                    "--query","[0].appId","-o","json"]) or ""
    APP_INSIGHTS_APP_ID = comp if isinstance(comp,str) else ""

assert SUBSCRIPTION_ID and APIM_NAME and RESOURCE_GROUP, "❌ az login / APIM 설정을 확인하세요."
ARM_BASE = (f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
            f"/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.ApiManagement/service/{APIM_NAME}")
CHAT_URL = f"{APIM_URL}/openai/deployments/{DEPLOYMENT_NAME}/chat/completions"
RUN_ID   = datetime.now(timezone.utc).strftime("%m%d%H%M%S")

def arm(method, path, body=None, query=None):
    url = f"{ARM_BASE}{path}?api-version={ARM_API}"
    args = ["rest","--method",method,"--url",url]
    if query: args += ["--query",query,"-o","json"]
    tmp = None
    if body is not None:
        tmp = tempfile.NamedTemporaryFile("w",suffix=".json",delete=False)
        json.dump(body,tmp); tmp.close()
        args += ["--headers","Content-Type=application/json","--body",f"@{tmp.name}"]
    out, err, rc = az(args)
    if tmp: os.unlink(tmp.name)
    return out, err, rc

def get_az_token():
    out, err, rc = az(["account","get-access-token","--resource",
                       "https://api.applicationinsights.io","--query","accessToken","-o","tsv"])
    return out.strip() if rc == 0 else ""
def query_ai(kql):
    if not APP_INSIGHTS_APP_ID: return None
    tok = get_az_token()
    if not tok: return None
    r = requests.post(f"https://api.applicationinsights.io/v1/apps/{APP_INSIGHTS_APP_ID}/query",
                      headers={"Authorization":f"Bearer {tok}","Content-Type":"application/json"},
                      json={"query":kql}, timeout=30)
    if r.status_code != 200:
        print(f"⚠️ KQL 실패 HTTP {r.status_code}: {r.text[:150]}"); return None
    return r.json()
def show(result):
    if not result or not result.get("tables") or not result["tables"][0]["rows"]:
        print("  결과 없음 (수집 지연 또는 데이터 없음)"); return 0
    t = result["tables"][0]; cols=[c["name"] for c in t["columns"]]; rows=t["rows"]
    w=[max(len(str(c)), max((len(str(r[i])) for r in rows), default=0)) for i,c in enumerate(cols)]
    print("  "+"  ".join(f"{c:<{w[i]}}" for i,c in enumerate(cols)))
    print("  "+"  ".join("-"*w[i] for i in range(len(cols))))
    for r in rows: print("  "+"  ".join(f"{str(v):<{w[i]}}" for i,v in enumerate(r)))
    return len(rows)

print("✅ 설정 완료")
print(f"   APIM: {APIM_NAME} | Model: {DEPLOYMENT_NAME} | RUN_ID: {RUN_ID}")
print(f"   App Insights App ID: {APP_INSIGHTS_APP_ID[:8]+'...' if APP_INSIGHTS_APP_ID else '(미설정)'}")


✅ 설정 완료
   APIM: apim-ai-gw-aigateway-20260716 | Model: gpt-4.1-nano | RUN_ID: 0716170856
   App Insights App ID: 0aea0792...


---
## 사전 활성화: 커스텀 메트릭 (metrics: true)

`llm-emit-token-metric` 이 App Insights `customMetrics` 로 데이터를 보내려면 APIM Diagnostics 의
`metrics` 플래그가 켜져 있어야 합니다. 아래 셀이 확인하고, 꺼져 있으면 켭니다.


In [2]:
# ─── 커스텀 메트릭 활성화 보장 ───
out, err, rc = arm("GET", "/diagnostics/applicationinsights", query="properties.metrics")
flag = (out or "").strip().strip('"').lower()
if flag == "true":
    print("✅ custom metrics 이미 활성 (metrics: true)")
else:
    full = az_json(["rest","--method","GET","--url",
                    f"{ARM_BASE}/diagnostics/applicationinsights?api-version={ARM_API}","-o","json"])
    if full and "properties" in full:
        props = full["properties"]; props["metrics"] = True
        arm("PUT", "/diagnostics/applicationinsights", body={"properties": props})
        print("✅ custom metrics 활성화 완료 (metrics: true 설정)")
    else:
        print("⚠️ diagnostic 을 찾지 못했습니다 — Portal 에서 수동 활성화하세요 (Lab 6 · 2단계)")
print("💡 Portal: App Insights → Usage and estimated costs → Custom metrics (Preview) → With dimensions 도 권장")


✅ custom metrics 이미 활성 (metrics: true)
💡 Portal: App Insights → Usage and estimated costs → Custom metrics (Preview) → With dimensions 도 권장


---
## Phase 1: 라벨 트래픽 생성 (2개 구독)

`team-a`(넉넉) / `team-b`(소한도) 구독으로 호출합니다. 각 요청은 `Subscription ID`·`Provider`·`Model`
차원으로 태깅되어 `customMetrics` 에 적재됩니다. `team-b` 는 `quota-by-key calls=4` 로 낮춰
**일부 요청이 403 으로 차단**되어 거버넌스 추이(Query 3)에 나타납니다.

> 프로바이더는 실제 배포된 `azure-openai` 로 라벨링합니다. Lab 8 통합 라우팅을 배포하면
> `openai`/`bedrock`/`anthropic`/`gemini` 가 **Provider 차원의 다른 값**으로 함께 나타납니다.


In [3]:
# ─── Phase 1: Product/구독 생성 + 라벨 트래픽 ───
def emit_metric(provider, model):
    return (f'<set-variable name="provider" value="{provider}" />'
            f'<set-variable name="modelName" value="{model}" />'
            f'<llm-emit-token-metric namespace="ai-gateway-metrics">'
            f'<dimension name="Subscription ID" value="@(context.Subscription.Id)" />'
            f'<dimension name="Provider" value="@((string)context.Variables[&quot;provider&quot;])" />'
            f'<dimension name="Model" value="@((string)context.Variables[&quot;modelName&quot;])" />'
            f'<dimension name="API ID" value="@(context.Api.Id)" />'
            f'<dimension name="Client IP" value="@(context.Request.IpAddress)" />'
            f'</llm-emit-token-metric>')
def policy(inner):
    return ("<policies><inbound><base />"+inner+"</inbound><backend><base /></backend>"
            "<outbound><base /></outbound><on-error><base /></on-error></policies>")

API_ID = None
for cand in ["multicloud-openai","azure-openai"]:
    o,e,rc = arm("GET", f"/apis/{cand}", query="name")
    if rc==0 and o: API_ID=cand; break
assert API_ID, "❌ API 없음"; print(f"▶ API: {API_ID}")

TEAMS = {
  "team-a": {"calls": 100000, "n": 6},
  "team-b": {"calls": 4,      "n": 6},   # 4 성공 후 403
}
SUBS = {}
for pid, cfg in TEAMS.items():
    arm("PUT", f"/products/{pid}", body={"properties":{"displayName":pid,
        "subscriptionRequired":True,"approvalRequired":False,"state":"published"}})
    arm("PUT", f"/products/{pid}/apis/{API_ID}")
    inner = (f'<quota-by-key counter-key="lab10-{RUN_ID}-{pid}" calls="{cfg["calls"]}" renewal-period="3600" />'
             + emit_metric("azure-openai", DEPLOYMENT_NAME))
    arm("PUT", f"/products/{pid}/policies/policy", body={"properties":{"format":"rawxml","value":policy(inner)}})
    sid = f"sub-{pid}-{RUN_ID}"
    arm("PUT", f"/subscriptions/{sid}", body={"properties":{"displayName":sid,
        "scope":f"{ARM_BASE}/products/{pid}","state":"active"}})
    o,e,rc = arm("POST", f"/subscriptions/{sid}/listSecrets", query="primaryKey")
    SUBS[pid] = {"sid": sid, "key": (o or "").strip().strip('"')}
    print(f"  ✅ {pid}: Product+구독 생성")

print("\n⏳ 전파 대기 60초..."); time.sleep(60)

print("\n▶ 트래픽 생성")
for pid, cfg in TEAMS.items():
    ok=blocked=0
    for i in range(cfg["n"]):
        r = requests.post(CHAT_URL, params={"api-version":API_VERSION},
            headers={"Content-Type":"application/json","Ocp-Apim-Subscription-Key":SUBS[pid]["key"]},
            json={"messages":[{"role":"user","content":"observability test ping"}],"max_tokens":20}, timeout=30)
        if r.status_code==200: ok+=1
        elif r.status_code==403: blocked+=1
        time.sleep(0.4)
    print(f"  {pid}: 200×{ok}, 403×{blocked}")
print("✅ Phase 1 완료 — 메트릭 전송됨")


▶ API: azure-openai


  ✅ team-a: Product+구독 생성


  ✅ team-b: Product+구독 생성

⏳ 전파 대기 60초...



▶ 트래픽 생성


  team-a: 200×6, 403×0


  team-b: 200×5, 403×1
✅ Phase 1 완료 — 메트릭 전송됨


---
## Phase 2: 수집 대기

`customMetrics` 는 보통 **5~10분** 후 조회됩니다. 아래 셀이 데이터가 보일 때까지 폴링합니다
(최대 ~8분). `requests` 테이블(429/403)은 더 빨리(2~5분) 나타납니다.


In [4]:
# ─── Phase 2: customMetrics 수집 폴링 ───
POLL_KQL = """customMetrics
| where timestamp > ago(1h)
| where name == "Total Tokens"
| where customDimensions["Subscription ID"] startswith "sub-"
| summarize c=count()"""
ready=False
for attempt in range(1, 9):
    res = query_ai(POLL_KQL)
    n = 0
    if res and res.get("tables") and res["tables"][0]["rows"]:
        n = res["tables"][0]["rows"][0][0]
    print(f"  [{attempt}] customMetrics rows={n}")
    if n and int(n) > 0: ready=True; break
    if attempt < 8: time.sleep(60)
print("✅ 수집 확인" if ready else "⚠️ 아직 미수집 — Query 3(requests)는 조회 가능, 나머지는 잠시 후 재실행")


  [1] customMetrics rows=0


  [2] customMetrics rows=0


  [3] customMetrics rows=7
✅ 수집 확인


---
## Phase 3: KQL 실시간 조회

App Insights REST API(`az` 토큰 인증)로 아래 쿼리를 실행합니다.


In [5]:
# ─── Query 1: Provider × Subscription × Model 토큰 ───
print("═"*70); print(" Query 1: 구독·프로바이더·모델별 토큰 사용량"); print("═"*70)
Q1 = """customMetrics
| where timestamp > ago(1h)
| where name in ("Total Tokens","Prompt Tokens","Completion Tokens")
| extend sub=tostring(customDimensions["Subscription ID"]),
         provider=tostring(customDimensions["Provider"]),
         model=tostring(customDimensions["Model"])
| where sub startswith "sub-"
| summarize totalTokens=sumif(value,name=="Total Tokens"),
            prompt=sumif(value,name=="Prompt Tokens"),
            completion=sumif(value,name=="Completion Tokens")
    by sub, provider, model
| order by totalTokens desc"""
show(query_ai(Q1))


══════════════════════════════════════════════════════════════════════
 Query 1: 구독·프로바이더·모델별 토큰 사용량
══════════════════════════════════════════════════════════════════════


  sub                    provider      model         totalTokens  prompt  completion
  ---------------------  ------------  ------------  -----------  ------  ----------
  sub-team-a-0716170856  azure-openai  gpt-4.1-nano  186          66      120       
  sub-team-b-0716170856  azure-openai  gpt-4.1-nano  155          55      100       


2

In [6]:
# ─── Query 2: 구독별 비용 추정 (프로바이더 단가 예시) ───
print("═"*70); print(" Query 2: 구독별 크로스클라우드 비용 추정"); print("═"*70)
Q2 = """customMetrics
| where timestamp > ago(1h)
| where name == "Total Tokens"
| extend sub=tostring(customDimensions["Subscription ID"]),
         provider=tostring(customDimensions["Provider"])
| where sub startswith "sub-"
| summarize tokens=sum(value) by sub, provider
| extend estCostUsd = round(case(
    provider=="bedrock", tokens*0.000003,
    provider=="anthropic", tokens*0.000003,
    provider=="gemini", tokens*0.0000005,
    provider=="openai", tokens*0.000005,
    tokens*0.000002), 6)
| order by estCostUsd desc"""
show(query_ai(Q2))
print("\n  ※ 단가는 예시값 — 실제 요율은 각 프로바이더 콘솔 기준")


══════════════════════════════════════════════════════════════════════
 Query 2: 구독별 크로스클라우드 비용 추정
══════════════════════════════════════════════════════════════════════


  sub                    provider      tokens  estCostUsd
  ---------------------  ------------  ------  ----------
  sub-team-a-0716170856  azure-openai  186     0.000372  
  sub-team-b-0716170856  azure-openai  155     0.00031   

  ※ 단가는 예시값 — 실제 요율은 각 프로바이더 콘솔 기준


In [7]:
# ─── Query 3: 429/403 거버넌스 차단 추이 (requests, 자동 수집) ───
print("═"*70); print(" Query 3: 거버넌스 차단(429/403) 추이"); print("═"*70)
Q3 = """requests
| where timestamp > ago(1h)
| where resultCode in ("429","403","401")
| summarize count() by resultCode, bin(timestamp, 5m)
| order by timestamp desc"""
show(query_ai(Q3))
print("\n  💡 403=quota 차단(토큰/요청), 429=TPM 차단, 401=구독 키 없음")


══════════════════════════════════════════════════════════════════════
 Query 3: 거버넌스 차단(429/403) 추이
══════════════════════════════════════════════════════════════════════


  resultCode  timestamp             count_
  ----------  --------------------  ------
  403         2026-07-16T17:10:00Z  1     
  403         2026-07-16T16:45:00Z  1     
  401         2026-07-16T16:40:00Z  1     
  429         2026-07-16T16:40:00Z  1     
  403         2026-07-16T16:40:00Z  1     
  401         2026-07-16T16:30:00Z  5     
  403         2026-07-16T16:30:00Z  3     

  💡 403=quota 차단(토큰/요청), 429=TPM 차단, 401=구독 키 없음


---
### 📋 저장된 KQL 쿼리 모음 — Portal(App Insights → Logs)

**1) 구독·프로바이더·모델별 토큰**
```kql
customMetrics
| where timestamp > ago(24h)
| where name in ("Total Tokens","Prompt Tokens","Completion Tokens")
| extend sub=tostring(customDimensions["Subscription ID"]),
         provider=tostring(customDimensions["Provider"]),
         model=tostring(customDimensions["Model"])
| summarize totalTokens=sumif(value,name=="Total Tokens") by sub, provider, model
| order by totalTokens desc
```

**2) 구독별 TPM(분당 토큰) 시계열**
```kql
customMetrics
| where timestamp > ago(1h)
| where name == "Total Tokens"
| extend sub=tostring(customDimensions["Subscription ID"])
| summarize TPM=sum(value) by sub, bin(timestamp, 1m)
| render timechart
```

**3) 구독별 월 토큰 quota 사용률**
```kql
let quota = 2000000;   // team-b token-quota (프로덕션 값)
customMetrics
| where timestamp > ago(30d)
| where name == "Total Tokens"
| extend sub=tostring(customDimensions["Subscription ID"])
| summarize used=sum(value) by sub
| extend quota=quota, usedPct=round(100.0*used/quota, 2)
```

**4) 거버넌스 차단(429/403) 추이**
```kql
requests
| where timestamp > ago(1h)
| where resultCode in ("429","403")
| summarize count() by resultCode, bin(timestamp, 5m)
| render columnchart
```

### 🔭 구독별 거버넌스 Workbook — 원커맨드 배포

위 KQL 을 손으로 타일에 넣을 필요 없이, 배포 가능한 템플릿을 제공합니다.
상단 **시간범위 · 토큰 쿼터 기준 · 구독(멀티선택)** 필터에 연동되는 **8개 타일**
(쿼터 사용률 · TPM · RPS · 프로바이더 분해 · 비용 · 429/403 차단 · SLO · 프롬프트 감사)
로 멀티클라우드 × 구독을 한 평면에서 리포팅합니다.

```bash
cd labs/lab10-governance-observability
RESOURCE_GROUP=<APIM 이 있는 RG> ./deploy-workbook.sh
# → App Insights 자동 탐색 → Microsoft.Insights/workbooks 배포(멱등) → 포털 딥링크 출력
```

- 템플릿: [`workbook-template.json`](./workbook-template.json)
- 배포 스크립트: [`deploy-workbook.sh`](./deploy-workbook.sh)
- 타일 KQL 전문·파라미터 설명: [Lab 10 README · 5단계](./README.md)


---
## 정리
테스트로 만든 Product/구독을 삭제합니다. (App Insights 에 적재된 메트릭은 보존됩니다.)


In [8]:
# ─── 정리 ───
for pid, s in SUBS.items():
    arm("DELETE", f"/subscriptions/{s['sid']}")
for pid in TEAMS:
    az(["rest","--method","DELETE","--url",
        f"{ARM_BASE}/products/{pid}?api-version={ARM_API}&deleteSubscriptions=true"])
    print(f"  🧹 삭제: {pid}")
print("✅ 정리 완료")


  🧹 삭제: team-a


  🧹 삭제: team-b
✅ 정리 완료


---
## ✅ 요약

- **하나의 게이트웨이**가 여러 구독·프로바이더의 토큰·비용·차단을 단일 관측 평면에서 분해
- `llm-emit-token-metric` 의 `Subscription ID`·`Provider`·`Model` 차원 → 멀티 테넌트/멀티 클라우드 관측
- `customMetrics`(토큰/비용) + `requests`(429/403 거버넌스) 를 KQL 로 결합
- 저장된 쿼리를 Azure Monitor **Workbook/Dashboard/Alert** 로 승격 → 운영 관측
- 커스텀 메트릭은 **사전 활성화(metrics:true)** 필요 (이 노트북이 자동 처리)

→ [Lab 11: 리소스 정리](../lab11-cleanup/README.md) · [Lab 10 README](./README.md)
